# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

import os
from typing import List
from dotenv import load_dotenv

from lib.agents import Agent
from lib.llm import LLM
from lib.state_machine import Run
from lib.messages import BaseMessage
from lib.tooling import tool
from lib.vector_db import VectorStoreManager, CorpusLoaderService
from lib.rag import RAG
from lib.evaluation import TestCase, AgentEvaluator, EvaluationResult, EvaluationReport, PydanticOutputParser

In [3]:
# TODO: Load environment variables
import os
load_dotenv(dotenv_path="config.env")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
print(TAVILY_API_KEY)
print(OPENAI_API_KEY)
print(os.getenv("OPENAI_BASE_URL"))

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
voc-71328759916886552416536a5a3865718728.78066438
https://openai.vocareum.com/v1


In [4]:
vector_store_manager = VectorStoreManager(openai_api_key=OPENAI_API_KEY)
vector_store = vector_store_manager.get_or_create_store("udaplay")


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [5]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

def retrieve_game(query: str, vector_store, n_results: int = 3):
    """
    Semantic search: Finds most relevant results in the vector DB.

    Args:
        query: a question about the game industry.

    Returns:
        A list of matching game records.
    """
    results = vector_store._collection.query(
        query_texts=[query],
        n_results=n_results
    )

    matches = results.get("metadatas", [[]])[0]
    return matches if matches else []

    
@tool
def get_games(query: str):
    """
    Retrieve relevant game information from the internal vector database.
    """
    return retrieve_game(query, vector_store)



In [6]:
print(get_games("games published by Sony Computer Entertainment"))


[{'Description': 'An open-world superhero game that lets players swing through New York City as Spider-Man, battling iconic villains.', 'Publisher': 'Sony Interactive Entertainment', 'Platform': 'PlayStation 4', 'Genre': 'Action-adventure', 'YearOfRelease': 2018, 'Name': "Marvel's Spider-Man"}, {'YearOfRelease': 2010, 'Genre': 'Racing', 'Description': 'A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.', 'Publisher': 'Sony Computer Entertainment', 'Platform': 'PlayStation 3', 'Name': 'Gran Turismo 5'}, {'Genre': 'Racing', 'Platform': 'PlayStation 1', 'Description': 'A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.', 'YearOfRelease': 1997, 'Publisher': 'Sony Computer Entertainment', 'Name': 'Gran Turismo'}]


#### Evaluate Retrieval Tool

In [7]:
import json

def evaluate_retrieval(question: str, retrieved_docs: list):
    llm_judge = LLM(model="gpt-4o-mini")

    judge_prompt = f"""
        Your task is to evaluate whether the retrieved documents are enough to answer the user's question.

        User question:
        {question}

        Retrieved documents:
        {retrieved_docs}

        Respond in JSON only:
        {{
        "useful": true or false,
        "description": "short explanation"
        }}
        """

    try:
        judge_response = llm_judge.invoke([UserMessage(content=judge_prompt)])
        content = (judge_response.content or "").strip()

        try:
            parsed = json.loads(content)
            return {
                "useful": parsed.get("useful", len(retrieved_docs) > 0),
                "description": parsed.get("description", "No description returned.")
            }
        except json.JSONDecodeError:
            lowered = content.lower()
            useful_guess = len(retrieved_docs) > 0

            if '"useful": false' in lowered or "useful: false" in lowered:
                useful_guess = False
            elif '"useful": true' in lowered or "useful: true" in lowered:
                useful_guess = True

            return {
                "useful": useful_guess,
                "description": content if content else "Judge returned non-JSON output."
            }

    except Exception as e:
        return {
            "useful": len(retrieved_docs) > 0,
            "description": f"Fallback evaluation due to error: {str(e)}"
        }


@tool
def judge_retrieval(question: str, retrieved_docs: list):
    return evaluate_retrieval(question, retrieved_docs)


In [8]:
docs = retrieve_game("games published by Sony Computer Entertainment", vector_store)
report = judge_retrieval(
    "games published by Sony Computer Entertainment",
    docs
)

print(report)
print(report["useful"])
print(report["description"])


{'useful': True, 'description': "The retrieved documents include games published by Sony Computer Entertainment, specifically 'Gran Turismo 5' and 'Gran Turismo', which directly answer the user's question."}
True
The retrieved documents include games published by Sony Computer Entertainment, specifically 'Gran Turismo 5' and 'Gran Turismo', which directly answer the user's question.


#### Game Web Search Tool

In [9]:

from typing import Dict
import os
from datetime import datetime
from tavily import TavilyClient
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

In [10]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

def web_search(query: str, search_depth: str = "advanced") -> Dict:
    """
    Search the web using Tavily API.

    args:
        query (str): Search query
        search_depth (str): Type of search - 'basic' or 'advanced'
    """
    api_key = os.getenv("TAVILY_API_KEY")
    client = TavilyClient(api_key=api_key)

    search_result = client.search(
        query=query,
        search_depth=search_depth,
        include_answer=True,
        include_raw_content=False,
        include_images=False
    )

    formatted_results = {
        "answer": search_result.get("answer", ""),
        "results": search_result.get("results", []),
        "search_metadata": {
            "timestamp": datetime.now().isoformat(),
            "query": query
        }
    }

    return formatted_results

@tool
def game_web_search(question: str) -> Dict:
    """
    Web search tool for game industry questions.

    args:
    - question: a question about game industry.
    """
    return web_search(question)

In [11]:
result = game_web_search("What are recent trends in the video game industry?")
print(result["answer"])
print(result["results"][:2])


Recent trends in the video game industry include consolidation among top companies, the rise of cloud and mobile gaming, and increased focus on player wellbeing and fair play. Generative AI and user-generated content are also driving innovation.
[{'url': 'https://www.juegostudio.com/blog/recent-trends-reforming-gaming-industry', 'title': 'Top Video Game Industry Trends for 2026 | Juego Studios', 'content': 'Key policy shifts:\n\n AI-generated asset labeling becomes legally required\n Loot box restrictions expand across Europe and emerging markets\n Fair-play certifications increase visibility in app stores\n Wellbeing tools—like fatigue detection or break prompts—become standard for live-service titles\n\nThese guardrails protect players while helping studios build sustainable long-term communities.\n\n## Conclusion\n\nThe most significant video game industry trends for 2026 revolve around intelligent automation, accessible design, alignment with global audiences, and seamless multi-de

### Agent

In [12]:
from lib.agents import Agent

instructions = """
You are a game industry research assistant.

You have access to these tools:
1. get_games: retrieve relevant game records from the internal vector database
2. judge_retrieval: evaluate whether retrieved records are sufficient to answer the question
3. game_web_search: search the web if retrieval is insufficient

Always follow this process:
- First call get_games with the user's question.
- Then call judge_retrieval with the original question and the retrieved documents.
- If judge_retrieval says useful=true, answer using the retrieved documents.
- If judge_retrieval says useful=false, call game_web_search and answer using the web results.
- Cite clearly whether your answer came from retrieval or web search.
- If neither source is sufficient, say you do not know.
"""


agent = Agent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=[get_games, judge_retrieval, game_web_search],
    temperature=0
)


In [13]:
run = agent.invoke(
    "When were Pokémon Gold and Silver released?",
    session_id="session_1"
)

final_state = run.get_final_state()

print("\n=== TOOLS CALLED ===")
for msg in final_state["messages"]:
    tool_calls = getattr(msg, "tool_calls", None)
    if tool_calls:
        for tc in tool_calls:
            print(f"Function(name='{tc.function.name}', arguments={tc.function.arguments})")

print("\nFinal answer:")
print(final_state["messages"][-1].content)


[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

=== TOOLS CALLED ===
Function(name='get_games', arguments={"query":"Pokémon Gold and Silver release date"})
Function(name='judge_retrieval', arguments={"question":"When were Pokémon Gold and Silver released?","retrieved_docs":"[{\"Genre\": \"Role-playing\", \"YearOfRelease\": 1999, \"Description\": \"Second-generation Pok\u0000mon games introducing new regions, Pok\u0000mon, and gameplay mechanics.\", \"Platform\": \"Game Boy Color\", \"Publisher\": \"Nintendo\", \"Name\": \"Pok\u0000mon Gold and Silver\"},{\"Publisher\": \"Nintendo\", \"Genre\": \"Role-playing\", \"Platform\": \"Game Boy Advance\", \"Name\": \"Pok\u0000mon Ruby and Sapphire\"

In [14]:
run2 = agent.invoke(
    "What platform were they released on?",
    session_id="session_1"
)

final_state2 = run2.get_final_state()

print("\n=== TOOLS CALLED ===")
for msg in final_state2["messages"]:
    tool_calls = getattr(msg, "tool_calls", None)
    if tool_calls:
        for tc in tool_calls:
            print(f"Function(name='{tc.function.name}', arguments={tc.function.arguments})")

print("\nFinal answer:")
print(final_state2["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

=== TOOLS CALLED ===
Function(name='get_games', arguments={"query":"Pokémon Gold and Silver release date"})
Function(name='judge_retrieval', arguments={"question":"When were Pokémon Gold and Silver released?","retrieved_docs":"[{\"Genre\": \"Role-playing\", \"YearOfRelease\": 1999, \"Description\": \"Second-generation Pok\u0000mon games introducing new regions, Pok\u0000mon, and gameplay mechanics.\", \"Platform\": \"Game Boy Color\", \"Publisher\": \"Nintendo\", \"Name\": \"Pok\u0000mon Gold and Silver\"},{\"Publisher\": \"Nintendo\", \"Genre\": \"Role-playing\", \"Platform\": \"Game Boy Advance\", \"Name\": \"Pok\u0000mon Ruby and Sapphire\", \"YearOfRelease\": 2002, \"Description\": \"Third-generation Pok\u0000mon games set in the Hoenn region, featuring new Pok\u0000mon and double battles.\"},{\"YearOfRelease\": 201

In [15]:
runs = agent.get_session_runs("session_1")
print("num saved states:", len(runs))
for i, state in enumerate(runs, 1):
    print(f"\nState {i}")
    print("total_tokens:", state.get("total_tokens"))
    for msg in state.get("messages", []):
        print("-", type(msg).__name__, getattr(msg, "content", None))


num saved states: 2

State 1
total_tokens: 1851
- SystemMessage 
You are a game industry research assistant.

You have access to these tools:
1. get_games: retrieve relevant game records from the internal vector database
2. judge_retrieval: evaluate whether retrieved records are sufficient to answer the question
3. game_web_search: search the web if retrieval is insufficient

Always follow this process:
- First call get_games with the user's question.
- Then call judge_retrieval with the original question and the retrieved documents.
- If judge_retrieval says useful=true, answer using the retrieved documents.
- If judge_retrieval says useful=false, call game_web_search and answer using the web results.
- Cite clearly whether your answer came from retrieval or web search.
- If neither source is sufficient, say you do not know.

- UserMessage When were Pokémon Gold and Silver released?
- AIMessage None
- ToolMessage [{"Genre": "Role-playing", "YearOfRelease": 1999, "Description": "Second

In [16]:
# Question 2
run3 = agent.invoke("Which one was the first 3D platformer Mario game?", session_id="session_2")
final_state3 = run3.get_final_state()

print("\n=== TOOLS CALLED ===")
for msg in final_state3["messages"]:
    tool_calls = getattr(msg, "tool_calls", None)
    if tool_calls:
        for tc in tool_calls:
            print(f"Function(name='{tc.function.name}', arguments={tc.function.arguments})")

print("\nFinal answer:")
print(final_state3["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

=== TOOLS CALLED ===
Function(name='get_games', arguments={"query":"first 3D platformer Mario game"})
Function(name='judge_retrieval', arguments={"question":"Which one was the first 3D platformer Mario game?","retrieved_docs":"[{\"Name\": \"Super Mario 64\", \"Genre\": \"Platformer\", \"Platform\": \"Nintendo 64\", \"Description\": \"A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.\", \"YearOfRelease\": 1996, \"Publisher\": \"Nintendo\"}, {\"YearOfRelease\": 1990, \"Genre\": \"Platformer\", \"Publisher\": \"Nintendo\", \"Name\": \"Super Mario World\", \"Platform\": \"Super N

In [17]:
# Question 2
run4 = agent.invoke("And what console was it released on?", session_id="session_2")
final_state4 = run4.get_final_state()

print("\n=== TOOLS CALLED ===")
for msg in final_state4["messages"]:
    tool_calls = getattr(msg, "tool_calls", None)
    if tool_calls:
        for tc in tool_calls:
            print(f"Function(name='{tc.function.name}', arguments={tc.function.arguments})")

print("\nFinal answer:")
print(final_state4["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

=== TOOLS CALLED ===
Function(name='get_games', arguments={"query":"first 3D platformer Mario game"})
Function(name='judge_retrieval', arguments={"question":"Which one was the first 3D platformer Mario game?","retrieved_docs":"[{\"Name\": \"Super Mario 64\", \"Genre\": \"Platformer\", \"Platform\": \"Nintendo 64\", \"Description\": \"A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.\", \"YearOfRelease\": 1996, \"Publisher\": \"Nintendo\"}, {\"YearOfRelease\": 1990, \"Genre\": \"Platformer\", \"Publisher\": \"Nintendo\", \"Name\": \"Super Mario World\", \"Platform\": \"Super Nintendo Entertainment System (SNES)\", \"Description\": \"A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.\"}, {\"Platf

In [18]:
# Question 3
run5 = agent.invoke("Was Mortal Kombat X released for Playstation 5?", session_id="session_3")
final_state5 = run5.get_final_state()

print("\n=== TOOLS CALLED ===")
for msg in final_state5["messages"]:
    tool_calls = getattr(msg, "tool_calls", None)
    if tool_calls:
        for tc in tool_calls:
            print(f"Function(name='{tc.function.name}', arguments={tc.function.arguments})")

print("\nFinal answer:")
print(final_state5["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

=== TOOLS CALLED ===
Function(name='get_games', arguments={"query":"Mortal Kombat X release for Playstation 5"})
Function(name='judge_retrieval', arguments={"question":"Was Mortal Kombat X released for Playstation 5?","retrieved_docs":"[{\"Genre\": \"Action-adventure\", \"Description\": \"The sequel to the acclaimed Spider-Man game, featuring both Peter Parker and Miles Morales as playable characters.\", \"Platform\": \"PlayStation 5\", \"Publisher\": \"Sony Interactive Entertainment\", \"YearOfRelease\": 2023, \"Name\": \"Marvel's Spider-Man 2\"}, {\"Pla

In [19]:

run6 = agent.invoke("What platforms was it released on originally?", session_id="session_3")
final_state6 = run6.get_final_state()

print("\n=== TOOLS CALLED ===")
for msg in final_state5["messages"]:
    tool_calls = getattr(msg, "tool_calls", None)
    if tool_calls:
        for tc in tool_calls:
            print(f"Function(name='{tc.function.name}', arguments={tc.function.arguments})")

print("\nFinal answer:")
print(final_state6["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

=== TOOLS CALLED ===
Function(name='get_games', arguments={"query":"Mortal Kombat X release for Playstation 5"})
Function(name='judge_retrieval', arguments={"question":"Was Mortal Kombat X released for Playstation 5?","retrieved_docs":"[{\"Genre\": \"Action-adventure\", \"Description\": \"The sequel to the acclaimed Spider-Man game, featuring both Peter Parker and Miles Morales as playable characters.\", \"Platform\": \"PlayStation 5\", \"Publisher\": \"Sony Interactive Entertainment\", \"YearOfRelease\": 2023, \"Name\": \"Marvel's Spider-Man 2\"}, {\"Pla

In [20]:

run7 = agent.invoke("Who published it?", session_id="session_3")
final_state7 = run7.get_final_state()

print("\n=== TOOLS CALLED ===")
for msg in final_state5["messages"]:
    tool_calls = getattr(msg, "tool_calls", None)
    if tool_calls:
        for tc in tool_calls:
            print(f"Function(name='{tc.function.name}', arguments={tc.function.arguments})")

print("\nFinal answer:")
print(final_state7["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

=== TOOLS CALLED ===
Function(name='get_games', arguments={"query":"Mortal Kombat X release for Playstation 5"})
Function(name='judge_retrieval', arguments={"question":"Was Mortal Kombat X released for Playstation 5?","retrieved_docs":"[{\"Genre\": \"Action-adventure\", \"Description\": \"The sequel to the acclaimed Spider-Man game, featuring both Peter Parker and Miles Morales as playable characters.\", \"Platform\": \"PlayStation 5\", \"Publisher\": \"Sony Interactive Entertainment\", \"YearOfRelease\": 2023, \"Name\": \"Marvel's Spider-Man 2\"}, {\"Pla

In [26]:
def print_turn_summary(agent, session_id: str):
    states = agent.get_session_runs(session_id)

    for turn_idx, state in enumerate(states, 1):
        print("\n" + "=" * 60)
        print(f"TURN {turn_idx}")
        print("=" * 60)

        user_msgs = [m for m in state["messages"] if type(m).__name__ == "UserMessage"]
        ai_msgs = [m for m in state["messages"] if type(m).__name__ == "AIMessage"]

        if user_msgs:
            print("Latest user message:")
            print(user_msgs[-1].content)

        print("\nTools called this turn:")
        for msg in state["messages"]:
            tool_calls = getattr(msg, "tool_calls", None)
            if tool_calls:
                for tc in tool_calls:
                    print(f"- {tc.function.name}")

        if ai_msgs:
            print("\nLatest assistant answer:")
            print(ai_msgs[-1].content)


In [27]:
print_turn_summary(agent, "session_3")



TURN 1
Latest user message:
Was Mortal Kombat X released for Playstation 5?

Tools called this turn:
- get_games
- judge_retrieval
- game_web_search

Latest assistant answer:
Mortal Kombat X was originally released for PlayStation 4 in 2015. However, it is playable on PlayStation 5 as well, and it is included in the PlayStation Plus Collection for PS5 subscribers. This means that while there is no dedicated PS5 version, players can still enjoy Mortal Kombat X on the new console. 

For more details, you can check the sources: [IGN](https://www.ign.com/games/mortal-kombat-x) and [Mortal Kombat Online](https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V).

TURN 2
Latest user message:
What platforms was it released on originally?

Tools called this turn:
- get_games
- judge_retrieval
- game_web_search
- get_games
- judge_retrieval
- game_web_search

Latest assistant answer:
Mortal Kombat X was originally released on the followin

### (Optional) Advanced

In [23]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes